In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import os

#load dataset
df = pd.read_csv("twitter_training.csv")
df = df.dropna()
df.isna().sum()
df.head()
#remove table header and add something more descriptive
df = df.drop(df.index[0])
#drop the first 2 columns...they're not needed
df = df.drop(df.columns[:1], axis=1)
df.head()
df.columns = ['Topic', 'Sentiment', 'Comment']
df.head()

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from collections import Counter
import re

# Encode target labels
label_encoder = LabelEncoder()
df['Sentiment_encoded'] = label_encoder.fit_transform(df['Sentiment'])
num_classes = len(label_encoder.classes_)
print(f"Target classes: {label_encoder.classes_}")

# Minimal text cleaning and tokenization
def tokenize(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    return text.split()

all_tokens = []
tokenized_comments = []
for comment in df['Comment']:
    tokens = tokenize(comment)
    tokenized_comments.append(tokens)
    all_tokens.extend(tokens)

# Build Vocabulary cleanly with dense 0-based indexing
vocab_counter = Counter(all_tokens)
# Keep words that appear at least 2 times
filtered_words = [word for word, count in vocab_counter.items() if count >= 2]

vocab = {'<PAD>': 0, '<UNK>': 1}
for i, word in enumerate(filtered_words):
    vocab[word] = i + 2

vocab_size = len(vocab)
print(f"Correctly mapped Vocabulary size: {vocab_size}")
print(f"Max token index: {max(vocab.values())}")


class TwitterDataset(Dataset):
    def __init__(self, tokenized_texts, labels, vocab, max_len=50):
        self.labels = labels
        self.max_len = max_len
        self.sequences = []

        for tokens in tokenized_texts:
            seq = [vocab.get(word, 1) for word in tokens] # 1 is <UNK>
            if len(seq) < max_len:
                seq = seq + [0] * (max_len - len(seq)) # Pad with 0
            else:
                seq = seq[:max_len] # Truncate
            self.sequences.append(seq)

        self.sequences = torch.tensor(self.sequences, dtype=torch.long)
        self.labels = torch.tensor(self.labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

# Split into train and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    tokenized_comments, df['Sentiment_encoded'].values, test_size=0.2, random_state=42
)

train_dataset = TwitterDataset(train_texts, train_labels, vocab)
val_dataset = TwitterDataset(val_texts, val_labels, vocab)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, num_layers=1):
        super(SentimentLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(0.3)

    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        lstm_out, (hidden, cell) = self.lstm(embedded)
        # Concatenate the final forward and backward hidden states
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        out = self.fc(self.dropout(hidden))
        return out

# Instantiate model parameters with the corrected vocab_size
embedding_dim = 100
hidden_dim = 128

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = SentimentLSTM(vocab_size, embedding_dim, hidden_dim, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10

for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for texts, labels in train_loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * texts.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss /= len(train_loader.dataset)
    train_acc = correct / total

    # Validation phase
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for texts, labels in val_loader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * texts.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_loss /= len(val_loader.dataset)
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc*100:.2f}%")


Target classes: ['Irrelevant' 'Negative' 'Neutral' 'Positive']
Correctly mapped Vocabulary size: 27265
Max token index: 27264
Using device: cpu
Epoch 1/10 | Train Loss: 1.2012 - Train Acc: 46.22% | Val Loss: 1.0388 - Val Acc: 57.75%
Epoch 2/10 | Train Loss: 0.9770 - Train Acc: 59.79% | Val Loss: 0.8772 - Val Acc: 64.57%
Epoch 3/10 | Train Loss: 0.8304 - Train Acc: 66.87% | Val Loss: 0.7686 - Val Acc: 69.46%
Epoch 4/10 | Train Loss: 0.7230 - Train Acc: 71.69% | Val Loss: 0.6621 - Val Acc: 74.28%
Epoch 5/10 | Train Loss: 0.6343 - Train Acc: 75.43% | Val Loss: 0.5873 - Val Acc: 77.59%
Epoch 6/10 | Train Loss: 0.5625 - Train Acc: 78.53% | Val Loss: 0.5563 - Val Acc: 79.25%
Epoch 7/10 | Train Loss: 0.5012 - Train Acc: 80.91% | Val Loss: 0.4953 - Val Acc: 81.73%
Epoch 8/10 | Train Loss: 0.4579 - Train Acc: 82.65% | Val Loss: 0.4825 - Val Acc: 82.19%
Epoch 9/10 | Train Loss: 0.4157 - Train Acc: 84.18% | Val Loss: 0.4616 - Val Acc: 83.28%
Epoch 10/10 | Train Loss: 0.3823 - Train Acc: 85.62% | 

In [2]:
# Save the trained model, vocabulary, and label classes in the current directory
save_path = os.path.join(os.getcwd(), "sentiment_lstm_checkpoint.pth")

torch.save({
    "model_state_dict": model.state_dict(),
    "vocab": vocab,
    "label_classes": label_encoder.classes_.tolist(),
    "embedding_dim": embedding_dim,
    "hidden_dim": hidden_dim,
    "num_classes": num_classes,
    "max_len": 50,
}, save_path)

print(f"Model saved successfully to: {save_path}")


Model saved successfully to: C:\Users\HP PC\Downloads\PythonProject\TwitterSentimentAnalysis\sentiment_lstm_checkpoint.pth
